## Py11. Transform Payments Data — objective

- Compare raw fields with typed records and parsing failures.
- Interpret selected fields, samples and counts at their actual output grain.

Historical results; no code executed.
- [Original inputs: gizmobox/payments](../../../data/gizmobox/payments/) — 4 files.

# Py11. Transform Payments Data

Read the original source and historical results; nothing was executed.

[Full original payload](../../../artifacts/notebook-payloads/dea03-etl-with-apache-spark/Py11.%20Transform%20Payments%20Data.json)

## Transform Payments Data
1. Extract Date and Time from payment_timestamp and create new columns payment_date and payment_time
1. Map payment_status to contain descriptive values  
   (1-Success, 2-Pending, 3-Cancelled, 4-Failed)
1. Write transformed data to the Silver schema 

In [ ]:
df = spark.table('gizmobox.bronze.py_payments')
display(df)

payment_id,order_id,payment_timestamp,payment_status,payment_method
34,12,2024-12-06T22:30:12Z,1,PayPal
35,13,2024-12-17T05:25:34Z,2,PayPal
36,16,2024-12-09T10:45:56Z,4,Bank Transfer
37,17,2024-12-02T13:50:20Z,1,Bank Transfer
38,24,2024-12-22T08:30:55Z,1,PayPal
39,27,2024-12-03T11:45:30Z,2,Credit Card
40,28,2024-12-27T05:40:10Z,4,Credit Card
41,32,2024-12-29T09:20:40Z,1,PayPal
42,37,2024-12-23T12:10:05Z,2,Credit Card
43,38,2024-12-26T20:15:15Z,1,Credit Card


### 1. Extract Date and Time from payment_timestamp

In [ ]:
from pyspark.sql import functions as F

df_extracted_payments = (
                            df
                            .select (
                                'payment_id',
                                'order_id',
                                F.date_format('payment_timestamp','yyyy-MM-dd').cast('date').alias('payment_date'),
                                F.date_format('payment_timestamp','HH-mm-ss').alias('payment_time'),
                                'payment_status',
                                'payment_method'
                            )
)
display (df_extracted_payments)

payment_id,order_id,payment_date,payment_time,payment_status,payment_method
34,12,2024-12-06,22-30-12,1,PayPal
35,13,2024-12-17,05-25-34,2,PayPal
36,16,2024-12-09,10-45-56,4,Bank Transfer
37,17,2024-12-02,13-50-20,1,Bank Transfer
38,24,2024-12-22,08-30-55,1,PayPal
39,27,2024-12-03,11-45-30,2,Credit Card
40,28,2024-12-27,05-40-10,4,Credit Card
41,32,2024-12-29,09-20-40,1,PayPal
42,37,2024-12-23,12-10-05,2,Credit Card
43,38,2024-12-26,20-15-15,1,Credit Card


### 2. Map payment_status to contain descriptive values  
   (1-Success, 2-Pending, 3-Cancelled, 4-Failed)

In [ ]:
df_mapped_payments = (
                        df_extracted_payments
                        .select (
                            'payment_id',
                            'order_id',
                            'payment_date',
                            'payment_time',
                            F.when(df_extracted_payments.payment_status == 1, 'Success')
                             .when(df_extracted_payments.payment_status == 2, 'Pending')
                             .when(df_extracted_payments.payment_status == 3, 'Cancelled')
                             .when(df_extracted_payments.payment_status == 4, 'Failed')
                             .alias('payment_status'),
                             'payment_method'
                             )
)
display(df_mapped_payments)                        

payment_id,order_id,payment_date,payment_time,payment_status,payment_method
34,12,2024-12-06,22-30-12,Success,PayPal
35,13,2024-12-17,05-25-34,Pending,PayPal
36,16,2024-12-09,10-45-56,Failed,Bank Transfer
37,17,2024-12-02,13-50-20,Success,Bank Transfer
38,24,2024-12-22,08-30-55,Success,PayPal
39,27,2024-12-03,11-45-30,Pending,Credit Card
40,28,2024-12-27,05-40-10,Failed,Credit Card
41,32,2024-12-29,09-20-40,Success,PayPal
42,37,2024-12-23,12-10-05,Pending,Credit Card
43,38,2024-12-26,20-15-15,Success,Credit Card


### 3. Write transformed data to the Silver schema 

In [ ]:
df_mapped_payments.writeTo('gizmobox.silver.py_payments').createOrReplace()

Archived empty results: 1. Full archived values are in the linked payload reference in the notebook header.

Archived Databricks source — shown for reading, not Python execution.

```sql
%sql
SELECT * FROM gizmobox.silver.payments;
```

In [ ]:
# Archived output only; no computation.

payment_id,order_id,payment_date,payment_time,payment_status,payment_method
65,2,2025-01-19,05:05:13,Pending,PayPal
66,3,2025-01-08,23:45:00,Failed,Bank Transfer
67,4,2025-01-05,10:49:26,Success,Bank Transfer
68,8,2025-01-02,16:11:15,Pending,PayPal
69,14,2025-01-01,22:22:26,Failed,PayPal
70,18,2025-01-22,23:12:08,Success,Credit Card
71,22,2025-01-01,18:49:19,Success,PayPal
72,26,2025-01-05,22:01:25,Pending,Bank Transfer
73,29,2025-01-18,05:05:04,Failed,Bank Transfer
74,34,2025-01-12,11:30:15,Success,Credit Card
